In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

from sklearn.utils import check_random_state
import pandas as pd
import numpy as np
from logging import getLogger
from pathlib import Path
import os
import sys
sys.path.append(os.pardir)

import conf
from dataset import generate_synthetic_data
from utils import CoeffLoggedData, sample_set_of_actions
from opl import run_opl

# Set seeds to make the experiment reproducible
random_state = conf.random_state
random_ = check_random_state(random_state)

In [ ]:
logger = getLogger(__name__)
logger.info(f"The current working directory is {Path().cwd()}")

# log path
log_path = Path("../result/num_actions")
df_path = log_path / "df"
df_path.mkdir(exist_ok=True, parents=True)

In [ ]:

num_actions_list = []
for percent in conf.num_new_actions_percentage_list:
    num_actions_list.insert(0, int(conf.num_actions_all * (100-percent)/100))

num_new_actions_list = conf.num_actions_all - np.array(num_actions_list)

# Initialize the parameters we do not vary in this experiment
x = "num_actions"
xticks = num_new_actions_list
xticklabels = num_new_actions_list

# Set the coefficients for the data generating process
coef_dataset = CoeffLoggedData(dim_context = conf.dim_context, 
                            dim_feature = conf.dim_feature, 
                            num_id_per_each_feature = conf.num_id_per_each_feature, 
                            s = conf.s, 
                            random_state = conf.random_state)


In [ ]:
all_data = pd.DataFrame([], columns = ["seed", x, "method", "V", "existing_action_ratio", "new_action_ratio", "existing_V", "new_V", "per_existing_V", "per_new_V", "estimated_V"])

for num_actions in num_actions_list:
    print(f"[num_actions={num_actions}]")


    # Sample the set of actions A so that the logging policy can satisfy PCS
    set_of_actions = sample_set_of_actions(conf.dim_feature, conf.num_id_per_each_feature, conf.num_actions_all, num_actions, conf.random_state, s = conf.s)

    # Generate the test data
    dataset_test_all, dataset_test_logging = generate_synthetic_data(
        random_state = conf.random_state, 
        num_data = conf.num_test, 
        dim_context = conf.dim_context, 
        num_actions_all = conf.num_actions_all, 
        dim_feature = conf.dim_feature, 
        num_id_per_each_feature = conf.num_id_per_each_feature, 
        coef_dataset = coef_dataset, 
        set_of_actions = set_of_actions, 
        beta = conf.beta, 
        gamma = conf.gamma, 
        eta = conf.eta, 
        s = conf.s,
        lambda_ = conf.lambda_, 
    )

    for seed in range(conf.num_seeds):
        print(f"[{seed}/{conf.num_seeds}]")

        # Generate the train data
        dataset_train_all, dataset_train_logging = generate_synthetic_data(
            random_state=random_state + seed + 1,
            num_data=conf.num_train,
            dim_context=conf.dim_context,
            num_actions_all=conf.num_actions_all,
            dim_feature=conf.dim_feature,
            num_id_per_each_feature=conf.num_id_per_each_feature,
            coef_dataset=coef_dataset,
            set_of_actions=set_of_actions,
            beta=conf.beta,
            gamma=conf.gamma,
            eta=conf.eta,
            s=conf.s,
            lambda_=conf.lambda_,
        )

        # Calculate the absolute and relative policy values, percentage of the new actions, and the effectiveness of the new actions by the OPL methods
        result_data = run_opl(
            dataset_train_all,
            dataset_test_all,
            dataset_train_logging,
            dataset_test_logging,
            set_of_actions,
            conf.num_actions_all,
            num_actions,
            conf.kappa,
            conf.random_state + seed,
        )

        result_data["seed"] = seed
        num_actions_percent = (conf.num_actions_all - num_actions) / conf.num_actions_all * 100

        result_data["num_actions"] = num_actions_percent
        all_data = pd.concat([all_data, result_data], ignore_index=True)


all_data.to_csv(df_path / "all_data_results.csv", index=False)


In [ ]:
dataset_train_all.keys()